Libraries

In [5]:
import os
import shutil
import cv2

Paths

In [6]:
MOT_ROOT = "MOT20"
YOLO_ROOT = "MOT20_YOLO"

Split Def

In [10]:
splits = {
    "train": ["MOT20-01", "MOT20-02", "MOT20-03"],
    "val":  ["MOT20-05"]
}

for split in splits:
    os.makedirs(f"{YOLO_ROOT}/images/{split}", exist_ok=True)
    os.makedirs(f"{YOLO_ROOT}/labels/{split}", exist_ok=True)

Converting to YOLO Format

In [11]:
def convert_bbox(x, y, w, h, img_w, img_h):
    xc = (x + w / 2) / img_w
    yc = (y + h / 2) / img_h
    w /= img_w
    h /= img_h
    return xc, yc, w, h

for split, seqs in splits.items():
    for seq in seqs:
        seq_path = os.path.join(MOT_ROOT, "train", seq)
        img_path = os.path.join(seq_path, "img1")
        gt_path  = os.path.join(seq_path, "gt", "gt.txt")

        print(f"Processing {seq} ({split})")

        gt_data = {}
        with open(gt_path) as f:
            for line in f:
                frame, _, x, y, w, h, conf, cls, _ = line.strip().split(',')
                if int(conf) == 0:
                    continue
                frame = int(frame)
                gt_data.setdefault(frame, []).append(
                    [float(x), float(y), float(w), float(h)]
                )

        for img_name in sorted(os.listdir(img_path)):
            if not img_name.endswith(".jpg"):
                continue

            frame_id = int(img_name.split('.')[0])
            img_file = os.path.join(img_path, img_name)
            img = cv2.imread(img_file)
            h_img, w_img = img.shape[:2]

            label_file = os.path.join(
                YOLO_ROOT, "labels", split, img_name.replace(".jpg", ".txt")
            )

            with open(label_file, "w") as lf:
                for box in gt_data.get(frame_id, []):
                    xc, yc, bw, bh = convert_bbox(*box, w_img, h_img)
                    lf.write(f"0 {xc} {yc} {bw} {bh}\n")

            shutil.copy(
                img_file,
                os.path.join(YOLO_ROOT, "images", split, img_name)
            )

print("✅ MOT20 → YOLO (train/val) conversion complete")

✅ MOT20 → YOLO (train/val) conversion complete
